# 02 — Transformacao: Bronze → Silver

Transformacao dos dados brutos (Bronze) para a camada curada (Silver) com tipagem orientada por metadados e deduplicacao.

**Estrategia**:
- **Type casting metadata-driven**: Conversao de tipos guiada pelo dicionario `TYPE_CONFIG` (config.py)
- **Deduplicacao**: Por chave primaria composta + `_data_inclusao` DESC (window function ROW_NUMBER). Todas as tabelas possuem PK definida na tabela de metadados (`config.metadados_tabelas`).
- **Limpeza de strings**: `trim()` + `upper()` em colunas string
- **Formato de escrita**: Delta Lake com ACID e schema evolution

**Equivalente Fabric**: `2-metadados/ajustes-tipagem-deduplicacao.py`

**Hackathon PoD Academy** — Claro + Oracle

In [ ]:
import sys
sys.path.insert(0, "..")
from config import *

from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, trim, upper, to_date, to_timestamp,
    row_number, desc, current_timestamp, regexp_replace
)
from pyspark.sql import types

print(f"NAMESPACE: {NAMESPACE}")
print(f"Bronze bucket: {BUCKETS['bronze']}")
print(f"Silver bucket: {BUCKETS['silver']}")
print(f"Tabelas principais: {len(TABLES_MAIN)}")
print(f"Tabelas dimensao: {len(DIMENSION_TABLES)}")

In [ ]:
# Criar SparkSession com configuracao Delta Lake
builder = SparkSession.builder.appName("transform-bronze-silver")
for key, value in SPARK_DELTA_CONFIG.items():
    builder = builder.config(key, value)
spark = builder.getOrCreate()

print(f"SparkSession criada: {spark.version}")

In [ ]:
def type_cast(df, table_name, type_config):
    """Aplica type casting orientado por metadados (TYPE_CONFIG).
    
    Tipos suportados:
    - int, long: cast direto
    - double: cast direto
    - date: remove sufixo ':00:00:00' via regex, converte para DateType
    - timestamp_custom: formato ddMMMyyyy:HH:mm:ss (ex: 01MAY2024:00:00:00)
    - string: trim de espacos
    """
    if table_name not in type_config:
        print(f"  [WARN] Sem configuracao de tipos para '{table_name}' — schema preservado")
        return df
    
    col_types = type_config[table_name]
    cast_count = 0
    
    for col_name, col_type in col_types.items():
        if col_name not in df.columns:
            continue
        
        if col_type in ("int", "long", "double"):
            df = df.withColumn(col_name, col(col_name).cast(col_type))
        elif col_type == "date":
            # Remove sufixo :00:00:00 que vem de formatos SAS
            df = df.withColumn(col_name, regexp_replace(col(col_name).cast("string"), ":00:00:00", ""))
            df = df.withColumn(col_name, to_date(col(col_name)))
        elif col_type == "timestamp_custom":
            # Formato SAS: 01MAY2024:00:00:00 -> ddMMMyyyy:HH:mm:ss
            df = df.withColumn(col_name, to_timestamp(col(col_name).cast("string"), "ddMMMyyyy:HH:mm:ss"))
        elif col_type == "string":
            df = df.withColumn(col_name, trim(col(col_name).cast("string")))
        
        cast_count += 1
    
    print(f"  [TYPE_CAST] {table_name}: {cast_count} colunas convertidas")
    return df

In [ ]:
def deduplicate(df, pk_columns):
    """Deduplica DataFrame por chave primaria composta + _data_inclusao DESC.
    
    Todas as tabelas possuem PK definida na tabela de metadados (config.metadados_tabelas).
    Usa ROW_NUMBER particionado pela PK, ordenado por _data_inclusao DESC para manter
    o registro mais recente em caso de multiplas ingestoes.
    
    Fallback: se PK nao definida, faz dropDuplicates nas colunas de dados (exceto audit).
    """
    # Filtrar PKs que existem no DataFrame (seguranca contra schema mismatch)
    valid_pks = [c for c in pk_columns if c in df.columns]
    
    if valid_pks:
        if "_data_inclusao" in df.columns:
            window_spec = Window.partitionBy(*valid_pks).orderBy(desc("_data_inclusao"))
            df_ranked = df.withColumn("_rn", row_number().over(window_spec))
            before_count = df_ranked.count()
            df = df_ranked.filter(col("_rn") == 1).drop("_rn")
            after_count = df.count()
            print(f"  [DEDUP] PK={valid_pks}: {before_count:,} -> {after_count:,} ({before_count - after_count:,} duplicatas removidas)")
        else:
            before_count = df.count()
            df = df.dropDuplicates(valid_pks)
            after_count = df.count()
            print(f"  [DEDUP] PK={valid_pks} (dropDuplicates): {before_count:,} -> {after_count:,}")
        
        if len(valid_pks) < len(pk_columns):
            missing = set(pk_columns) - set(valid_pks)
            print(f"  [WARN] PKs ausentes no schema: {missing}")
    else:
        # Fallback: full-row dedup (exclui colunas de auditoria)
        data_cols = [c for c in df.columns if not c.startswith("_")]
        before_count = df.count()
        df = df.dropDuplicates(data_cols)
        after_count = df.count()
        print(f"  [DEDUP] Full-row (sem PK definida): {before_count:,} -> {after_count:,}")
    
    return df

In [ ]:
def read_table(spark, bucket, namespace, table_name, read_format="delta"):
    """Le tabela de um bucket OCI."""
    uri = f"oci://{bucket}@{namespace}/{table_name}/"
    if read_format == "delta":
        return spark.read.format("delta").load(uri)
    elif read_format == "parquet":
        return spark.read.parquet(uri)
    elif read_format == "csv":
        return spark.read.csv(uri, header=True, inferSchema=True, sep=";")
    else:
        raise ValueError(f"Formato nao suportado: {read_format}")


def write_table(df, bucket, namespace, path, write_format="delta", partition_by=None):
    """Escreve tabela em um bucket OCI."""
    uri = f"oci://{bucket}@{namespace}/{path}/"
    writer = df.write.mode("overwrite")
    if partition_by:
        writer = writer.partitionBy(partition_by)
    if write_format == "delta":
        writer.format("delta").option("overwriteSchema", "true").save(uri)
    else:
        writer.parquet(uri)
    print(f"  [WRITE] {uri} — formato: {write_format}")

## Fase 1: Tabelas Principais (7 tabelas)

Processamento das tabelas fato e base com type casting orientado por metadados (`TYPE_CONFIG`) e deduplicacao por chave primaria composta (conforme `config.metadados_tabelas` do Fabric).

| Tabela | PK (chave composta) | Tipo |
|--------|-----|------|
| `dados_cadastrais` | NUM_CPF, SAFRA, PROD | Base cadastral |
| `telco` | NUM_CPF, SAFRA, PROD | Comportamento telecom |
| `score_bureau_movel` | NUM_CPF, SAFRA | Targets e scores |
| `recarga` | NUM_CPF, DW_NUM_NTC, DAT_INSERCAO_CREDITO, HOR_INSERCAO_CREDITO, COD_GRUPO_CARTAO | Transacional — recargas |
| `pagamento` | NUM_CPF, CONTRATO, SEQ_FATURA, NUM_SUB_SEQ_FATURA, NUM_CREDITO_SEQ, COD_ATIVIDADE | Transacional — pagamentos |
| `faturamento` | NUM_CPF, DAT_REFERENCIA, NUM_FATURA_HASH, NUM_ENT_SEQ_FATURA, CONTRATO, COD_PLATAFORMA, DAT_STATUS_FAT | Transacional — faturamento |
| `dim_calendario` | DT_REFERENCIA | Dimensao calendario |

In [ ]:
# Processar tabelas principais: Bronze -> Silver
bronze_bucket = BUCKETS["bronze"]
silver_bucket = BUCKETS["silver"]

main_results = []

for table_name, table_cfg in TABLES_MAIN.items():
    print(f"\n{'='*60}")
    print(f"Processando: {table_name}")
    print(f"{'='*60}")
    
    # Ler Bronze
    df = read_table(spark, bronze_bucket, NAMESPACE, table_name, read_format=STORAGE_FORMAT)
    before_count = df.count()
    print(f"  Bronze: {before_count:,} linhas, {len(df.columns)} colunas")
    
    # Type casting
    df = type_cast(df, table_name, TYPE_CONFIG)
    
    # Limpeza de strings: trim + upper
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, types.StringType)]
    for c in string_cols:
        if not c.startswith("_"):
            df = df.withColumn(c, upper(trim(col(c))))
    print(f"  [CLEANSE] {len(string_cols)} colunas string limpas (trim + upper)")
    
    # Deduplicacao
    pk_columns = table_cfg["pk"]
    df = deduplicate(df, pk_columns)
    
    # Adicionar coluna de auditoria Silver
    df = df.withColumn("_data_alteracao_silver", current_timestamp())
    
    # Escrever Silver
    after_count = df.count()
    write_table(df, silver_bucket, NAMESPACE, f"rawdata/{table_name}", write_format=STORAGE_FORMAT)
    
    main_results.append({
        "table": table_name,
        "before": before_count,
        "after": after_count,
        "cols": len(df.columns),
    })
    print(f"  Resultado: {before_count:,} -> {after_count:,} linhas")

print(f"\n[SILVER] Fase 1 concluida: {len(main_results)} tabelas processadas")

## Fase 2: Tabelas de Dimensao (12 tabelas)

Processamento uniforme das tabelas de dimensao — **mesmo pipeline das tabelas principais**:
Bronze → type casting (metadados) → deduplicacao por PK + `_data_inclusao` DESC → Silver.

Equivalente ao Fabric: todas as 19 tabelas (7 principais + 12 dimensoes) passam pela mesma funcao `process_table()` no script `ajustes-tipagem-deduplicacao.py`.

| Tabela | PK | Fonte |
|--------|-----|-------|
| `canal_aquisicao_credito` | COD_CANAL_AQUISICAO | recarga_dim |
| `forma_pagamento` | DW_FORMA_PAGAMENTO | recarga_dim |
| `instituicao` | DW_INSTITUICAO | recarga_dim |
| `plano_preco` | DW_PLANO | recarga_dim |
| `plataforma` | COD_PLATAFORMA | recarga_dim |
| `promocao_credito` | COD_PROMOCAO | recarga_dim |
| `status_plataforma` | COD_STATUS_PLATAFORMA | recarga_dim |
| `tecnologia` | COD_TECNOLOGIA_DW | recarga_dim |
| `tipo_credito` | COD_TIPO_CREDITO | recarga_dim |
| `tipo_insercao` | DW_TIPO_INSERCAO | recarga_dim |
| `tipo_recarga` | DW_TIPO_RECARGA | recarga_dim |
| `tipo_faturamento` | DW_TIPO_FATURAMENTO | faturamento_dim |

In [ ]:
# Processar tabelas de dimensao: Bronze -> Silver (mesmo pipeline das principais)
# Equivale ao loop unico do Fabric sobre TODAS as tabelas de metadados.

dim_results = []

for dim_name, dim_cfg in DIMENSION_TABLES.items():
    print(f"\n{'='*60}")
    print(f"Processando dimensao: {dim_name}")
    print(f"{'='*60}")
    
    pk_columns = dim_cfg["pk"]
    source_folder = dim_cfg["source"]
    
    # Ler do Bronze (onde notebook 01 ja ingeriu com audit columns)
    try:
        df = read_table(spark, bronze_bucket, NAMESPACE, dim_name, read_format=STORAGE_FORMAT)
        print(f"  Bronze: {df.count():,} linhas, {len(df.columns)} colunas")
    except Exception as e:
        # Fallback: tentar ler do Bronze pelo source folder (nome alternativo)
        print(f"  [WARN] Falha ao ler '{dim_name}' do Bronze: {e}")
        try:
            df = read_table(spark, bronze_bucket, NAMESPACE, f"{source_folder}/{dim_name}", read_format=STORAGE_FORMAT)
            print(f"  Bronze (path alternativo): {df.count():,} linhas, {len(df.columns)} colunas")
        except Exception as e2:
            print(f"  [ERROR] Dimensao '{dim_name}' nao encontrada no Bronze — ignorada")
            dim_results.append({"table": dim_name, "before": 0, "after": 0, "cols": 0, "status": "ERRO"})
            continue
    
    before_count = df.count()
    
    # Type casting orientado por metadados (mesmo pipeline das tabelas principais)
    df = type_cast(df, dim_name, TYPE_CONFIG)
    
    # Limpeza de strings: trim + upper
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, types.StringType)]
    for c in string_cols:
        if not c.startswith("_"):
            df = df.withColumn(c, upper(trim(col(c))))
    print(f"  [CLEANSE] {len(string_cols)} colunas string limpas (trim + upper)")
    
    # Deduplicacao — mesma funcao das tabelas principais (PK + _data_inclusao DESC)
    df = deduplicate(df, pk_columns)
    
    # Adicionar coluna de auditoria Silver
    df = df.withColumn("_data_alteracao_silver", current_timestamp())
    
    # Escrever Silver
    after_count = df.count()
    write_table(df, silver_bucket, NAMESPACE, f"rawdata/{dim_name}", write_format=STORAGE_FORMAT)
    
    dim_results.append({
        "table": dim_name,
        "before": before_count,
        "after": after_count,
        "cols": len(df.columns),
        "status": "OK",
    })
    print(f"  Resultado: {before_count:,} -> {after_count:,} linhas")

print(f"\n[SILVER] Fase 2 concluida: {len(dim_results)} dimensoes processadas")

## Resumo e Validacao Silver

Resumo consolidado do processamento Bronze -> Silver com validacao automatizada:
- 19 tabelas existem (7 principais + 12 dimensoes)
- PK uniqueness por tabela
- Comparacao Bronze >= Silver rows (por dedup)

In [ ]:
# Resumo consolidado
print(f"{'='*70}")
print(f"RESUMO — Transformacao Bronze -> Silver")
print(f"{'='*70}")

print(f"\nTabelas Principais ({len(main_results)}):")
print(f"{'Tabela':30s} {'Bronze':>12s} {'Silver':>12s} {'Colunas':>8s}")
print("-" * 65)
for r in main_results:
    print(f"{r['table']:30s} {r['before']:>12,} {r['after']:>12,} {r['cols']:>8}")

print(f"\nTabelas de Dimensao ({len(dim_results)}):")
print(f"{'Tabela':30s} {'Bronze':>12s} {'Silver':>12s} {'Colunas':>8s} {'Status':>8s}")
print("-" * 75)
for r in dim_results:
    print(f"{r['table']:30s} {r['before']:>12,} {r['after']:>12,} {r.get('cols', 0):>8} {r.get('status', 'OK'):>8s}")

total_main = sum(r["after"] for r in main_results)
total_dim = sum(r["after"] for r in dim_results)

print(f"\n{'='*70}")
print(f"Total tabelas principais: {len(main_results):>3} tabelas, {total_main:>15,} linhas")
print(f"Total tabelas dimensao:   {len(dim_results):>3} tabelas, {total_dim:>15,} linhas")
print(f"Total geral:              {len(main_results) + len(dim_results):>3} tabelas, {total_main + total_dim:>15,} linhas")
print(f"Formato de armazenamento: {STORAGE_FORMAT}")
print(f"{'='*70}")

# Validacao Silver
print(f"\n{'='*60}")
print(f"VALIDACAO SILVER")
print(f"{'='*60}")
checks_pass = 0
checks_fail = 0
all_results = main_results + dim_results

# Check 1: 19 tabelas existem
expected_total = len(TABLES_MAIN) + len(DIMENSION_TABLES)
actual_total = len(all_results)
if actual_total == expected_total:
    print(f"  [PASS] {actual_total} tabelas processadas")
    checks_pass += 1
else:
    print(f"  [FAIL] {actual_total} tabelas (esperado: {expected_total})")
    checks_fail += 1

# Check 2: Row count > 0 para todas
empty = [r["table"] for r in all_results if r["after"] == 0]
if not empty:
    print(f"  [PASS] Todas as tabelas com linhas > 0")
    checks_pass += 1
else:
    print(f"  [FAIL] Tabelas vazias: {empty}")
    checks_fail += 1

# Check 3: Bronze >= Silver (dedup removing rows)
dedup_ok = True
for r in main_results:
    if r["before"] < r["after"]:
        print(f"  [FAIL] {r['table']}: Bronze ({r['before']:,}) < Silver ({r['after']:,})")
        dedup_ok = False
        checks_fail += 1
if dedup_ok:
    print(f"  [PASS] Bronze >= Silver para todas as tabelas principais (dedup consistente)")
    checks_pass += 1

# Check 4: PK uniqueness (sample check on dados_cadastrais)
from pyspark.sql.functions import col, count as spark_count
pk_check_table = "dados_cadastrais"
pk_cols = TABLES_MAIN[pk_check_table]["pk"]
uri = f"oci://{silver_bucket}@{NAMESPACE}/rawdata/{pk_check_table}/"
df_pk = spark.read.format(STORAGE_FORMAT).load(uri)
dup_count = df_pk.groupBy(*pk_cols).agg(spark_count("*").alias("cnt")).filter(col("cnt") > 1).count()
if dup_count == 0:
    print(f"  [PASS] PK uniqueness OK para {pk_check_table}")
    checks_pass += 1
else:
    print(f"  [FAIL] {dup_count:,} duplicatas em {pk_check_table}")
    checks_fail += 1

print(f"\n  Resultado: {checks_pass} PASS / {checks_fail} FAIL")
print(f"{'='*60}")